В цьому розділі представлено код для обробки, агрегації та збереження результатів в SQLite базу даних та CSV файл для подальшого використання в Tableau або інших інструментах візуалізації.

In [1]:
### Підключення до бази даних

import pandas as pd
import sqlite3

# 1. Підключення до SQLite бази даних
db_path = r"d:\VLAD\WORK\Projects\Portfolio\DA_Portfolio\DA\2025\global_mobile_reviews_project\data\processed\mobile_reviews.db"  
conn = sqlite3.connect(db_path)

print(f"Підключено до бази даних: {db_path}")

Підключено до бази даних: d:\VLAD\WORK\Projects\Portfolio\DA_Portfolio\DA\2025\global_mobile_reviews_project\data\processed\mobile_reviews.db


Витягування таблиці з бази даних

In [2]:
# 2. Витягуємо таблицю 'reviews' з бази даних в DataFrame для подальшої роботи
df_clean = pd.read_sql('SELECT * FROM reviews', conn)

# Перевірка першої частини даних
print(f"Перші 5 рядків даних з таблиці 'reviews':")
print(df_clean.head())

Перші 5 рядків даних з таблиці 'reviews':
   review_id      customer_name  age     brand          model  price_usd  \
0          1      Aryan Maharaj   45    Realme  Realme 12 Pro     337.31   
1          2  Davi Miguel Sousa   18    Realme  Realme 12 Pro     307.78   
2          3        Pahal Balay   27    Google        Pixel 6     864.53   
3          4       David Guzman   19    Xiaomi  Redmi Note 13     660.94   
4          5          Yago Leão   38  Motorola        Edge 50     792.13   

  price_local currency  exchange_rate_to_usd  rating  ... camera_rating  \
0   ₹27996.73      INR                 83.00       2  ...             1   
1   R$1754.35      BRL                  5.70       4  ...             2   
2   ₹71755.99      INR                 83.00       4  ...             5   
3  د.إ2425.65      AED                  3.67       3  ...             3   
4   R$4515.14      BRL                  5.70       3  ...             3   

  performance_rating design_rating display_rating 

## Агрегації та Обчислення
* Цінові Категорії
> Створюємо нову колонку price_category для класифікації ціни за категоріями (низька, середня, висока).

In [8]:
# 3.1 Цінові категорії (Low, Medium, High)
def price_category(price):
    if price < 200:
        return 'Low'
    elif price < 590:
        return 'Medium'
    else:
        return 'High'

df_clean['price_category'] = df_clean['price_usd'].apply(price_category)
print(df_clean['price_usd'].describe())
print(df_clean['price_category'].unique())

count    50000.000000
mean       689.693713
std        310.307331
min        180.020000
25%        450.792500
50%        637.040000
75%        900.975000
max       1499.890000
Name: price_usd, dtype: float64
['Medium' 'High' 'Low']


#### Агрегація за Брендом та Країною
> Для кожної пари "країна-бренд" обчислюємо середній рейтинг, кількість відгуків, кількість позитивних, негативних і нейтральних відгуків.

In [9]:
# 3.2 Агрегація за брендом та країною (середній рейтинг, кількість відгуків, позитивні/негативні відгуки)
agg_brand_country = df_clean.groupby(['country', 'brand']).agg(
    avg_rating=('rating', 'mean'),
    review_count=('review_id', 'count'),
    positive_count=('sentiment', lambda x: (x == 'positive').sum()),
    negative_count=('sentiment', lambda x: (x == 'negative').sum()),
    neutral_count=('sentiment', lambda x: (x == 'neutral').sum())
).reset_index()

#### Вікові Категорії
> Створюємо нову колонку age_category, що класифікує користувачів за віковими групами (підліток, молода людина, середній вік, старший вік).

In [11]:
# 3.3 Вікові категорії
def age_category(age):
    if age < 20:
        return 'Teen'
    elif age < 35:
        return 'Young Adult'
    elif age < 45:
        return 'Middle Age'
    else:
        return 'Senior'

df_clean['age_category'] = df_clean['age'].apply(age_category)
print(df_clean['age'].describe())
print(df_clean['age_category'].unique())

count    50000.000000
mean        30.075220
std          8.931307
min         18.000000
25%         23.000000
50%         29.000000
75%         36.000000
max         65.000000
Name: age, dtype: float64
['Senior' 'Teen' 'Young Adult' 'Middle Age']


#### Тренд Оцінок по Часу
> Додаємо нові колонки year і month для визначення року і місяця відгуку.

In [18]:
# 3.4 Тренд оцінок по часу (Рік і Місяць)
# Створюємо новий стовпець 'review_date_converted', перетворюємо його в datetime
df_clean['review_date_converted'] = pd.to_datetime(df_clean['review_date'], errors='coerce')
# Перевірка кількості NaT значень (відсутніх дат)
num_missing_dates = df_clean['review_date_converted'].isna().sum()
# Якщо відсутніх значень немає, оновлюємо оригінальний стовпець і видаляємо новий
if num_missing_dates == 0:
    df_clean['review_date'] = df_clean['review_date_converted']
    df_clean.drop(columns=['review_date_converted'], inplace=True)
# Якщо є відсутні значення, залишаємо новий стовпець і не змінюємо оригінальний
else:
    print(f"У стовпці 'review_date_converted' є {num_missing_dates} відсутніх значень.")
df_clean['year'] = df_clean['review_date'].dt.year
df_clean['month'] = df_clean['review_date'].dt.month
print ("Columns 'year' and 'month' created cucceessfully.")
#print(df_clean['year'].describe())
#print(df_clean['month'].describe())
# df_clean.info()

Columns 'year' and 'month' created cucceessfully.


#### Корисні Відгуки
> Створюємо колонку helpful_review, яка позначає, чи є відгук корисним на основі кількості голосів, більших за медіану.

In [20]:
# 3.5 Корисні відгуки: Створення колонки для найбільш корисних відгуків
df_clean['helpful_review'] = df_clean['helpful_votes'] > df_clean['helpful_votes'].median()
print (df_clean['helpful_review'].describe())
print (df_clean['helpful_review'].unique())

count     50000
unique        2
top       False
freq      26557
Name: helpful_review, dtype: object
[False  True]


### Збереження Результатів
#### Збереження в Базу Даних
> Оновлений DataFrame df_clean зберігається в нову таблицю reviews_with_aggregates в базі даних.

> Агреговані дані по бренду та країні зберігаються в таблиці agg_brand_country.

In [21]:
# 4. Зберігаємо оновлений DataFrame в нову таблицю в базі даних
df_clean.to_sql('reviews_with_aggregates', conn, if_exists='replace', index=False)
print ("Dataframe successfully exported to database.")

# Зберігаємо агреговані дані по бренду та країні в окрему таблицю
agg_brand_country.to_sql('agg_brand_country', conn, if_exists='replace', index=False)
print ("New table 'agg_brand_country' successfully created to database.")

Dataframe successfully exported to database.
New table 'agg_brand_country' successfully created to database.


#### Збереження в CSV
> Очищений та доповнений датасет зберігається в CSV файл для зручного використання в Tableau або інших інструментах візуалізації.

In [22]:
# 5. Зберігаємо DataFrame в CSV файл для подальшого використання в Tableau
csv_path = r'D:\VLAD\WORK\Projects\Portfolio\DA_Portfolio\DA\2025\global_mobile_reviews_project\data\processed\mobile_reviews_final.csv' 
df_clean.to_csv(csv_path, index=False)


#### Закриття З'єднання з БД

- Після виконання всіх операцій, з'єднання з базою даних закривається, щоб звільнити ресурси.

In [23]:
# Закриваємо з'єднання з базою даних
conn.close()

print(f"Збережено оновлений датасет в базу даних в таблиці 'reviews_with_aggregates'.")
print(f"Збережено агрегації в таблиці 'agg_brand_country'.")
print(f"Датасет також збережено в CSV файл за шляхом: {csv_path}")

Збережено оновлений датасет в базу даних в таблиці 'reviews_with_aggregates'.
Збережено агрегації в таблиці 'agg_brand_country'.
Датасет також збережено в CSV файл за шляхом: D:\VLAD\WORK\Projects\Portfolio\DA_Portfolio\DA\2025\global_mobile_reviews_project\data\processed\mobile_reviews_final.csv
